# 04 — Model validation

The three questions every model validator asks:

| Question | Tests |
|---|---|
| **Discrimination** — does it rank risk correctly? | AUC, Gini, KS, rank ordering |
| **Calibration** — is the PD level right? | calibration plot, Brier, Hosmer–Lemeshow, binomial test per grade |
| **Stability** — has the population moved? | PSI on the score, CSI per feature |

Most attention goes to the **out-of-time (OOT)** sample.

In [ ]:
import os, sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pdmodel import config as C
pd.set_option("display.float_format", "{:.4f}".format)
PROC = ROOT / "data" / "processed"
from pdmodel import plots, validation as V

In [ ]:
S = pd.read_pickle(PROC / "samples.pkl")
binner = pd.read_pickle(PROC / "binner.pkl")
M = pd.read_pickle(PROC / "model.pkl")
card, final = M["scorecard"], M["features"]
y  = {k: v["default"].to_numpy() for k, v in S.items()}
pd_ = {k: card.predict_pd(v) for k, v in S.items()}
sc = {k: card.score(v) for k, v in S.items()}

## 1. Discrimination

In [ ]:
disc = pd.DataFrame({k: V.discrimination(y[k], pd_[k]) for k in S}).T
disc["lc_subgrade_gini"] = [2 * V.benchmark_auc(y[k], S[k][C.BENCHMARK_COLUMN]) - 1 for k in S]
disc["rag"] = disc["gini"].map(V.rag_gini)
disc

In [ ]:
plots.roc_plot({k: (y[k], pd_[k]) for k in S}); plt.show()

In [ ]:
rank = V.rank_ordering(y["oot"], sc["oot"])
plots.bad_rate_by_band(rank, title="Default rate by score decile (OOT)"); plt.show()
rank

**Benchmark:** `lc_subgrade_gini` is how well LendingClub's own grade ranks the same loans.
LendingClub's grade used extra information (their full bureau pull), so matching it with a
handful of public features is a good result; beating it would be surprising — check for leakage.

## 2. Calibration

In [ ]:
cal = pd.DataFrame({k: V.calibration_summary(y[k], pd_[k]) for k in S}).T
cal

In [ ]:
plots.calibration_plot({k: V.calibration_table(y[k], pd_[k]) for k in S}); plt.show()

### Binomial test per rating grade
Grades are 7 equal-population score bands defined on train (grade 1 = best). For each grade we test
H0: true PD ≤ predicted PD. A red grade means the model **under-estimates** risk there.

In [ ]:
edges = V.grade_edges(sc["train"])
V.binomial_test_by_grade(y["oot"], pd_["oot"], V.assign_grade(sc["oot"], edges))

If OOT grades fail but rank ordering is fine, the problem is the **level**, not the ranking —
the usual fix is recalibration (re-estimate the intercept on recent vintages), not a full rebuild.

## 3. Stability

In [ ]:
print(f"Score PSI train->test: {V.psi(sc['train'], sc['test']):.4f}  ({V.rag_psi(V.psi(sc['train'], sc['test']))})")
print(f"Score PSI train->OOT : {V.psi(sc['train'], sc['oot']):.4f}  ({V.rag_psi(V.psi(sc['train'], sc['oot']))})")
csi = V.csi_by_feature(binner, S["train"], S["oot"], final)
plots.psi_bar(csi); plt.show()
csi

## 4. Write it up
Run `python scripts/run_pipeline.py` to regenerate every table and figure and the
auto-filled `reports/validation_report.md`. Then write the ✍️ commentary sections yourself —
**that write-up is the part interviewers will ask about.**

## ✍️ Interview prep
1. Gini is fine OOT but the binomial test fails in every grade. What's going on and what do you recommend?
2. Why is Hosmer–Lemeshow almost useless with 500k loans?
3. Score PSI is 0.08 but one feature has CSI 0.3. Is the model OK?
4. What would an ongoing monitoring plan for this model look like?